1. Setting Up and Scaling Features
For this analysis, we will transform our continuous target (annual_medical_cost) into a binary classification problem: predicting whether an individual is "High Cost" (above the 75th percentile of expenditures) or "Standard Cost".

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

# Load dataset
df = pd.read_csv("medical_insurance.csv")

# Define features
continuous_features = ["age", "income", "bmi", "visits_last_year", "risk_score", "claims_count"]
categorical_features = ["sex", "region", "urban_rural", "smoker", "plan_type"]
target_raw = "annual_medical_cost"

# Clean missing values
df_clean = df.dropna(subset=continuous_features + categorical_features + [target_raw])

# Define binary classification target: 1 if in the top 25% highest costs, else 0
threshold = df_clean[target_raw].quantile(0.75)
df_clean["is_high_cost"] = (df_clean[target_raw] > threshold).astype(int)
y = df_clean["is_high_cost"]

# Encode categorical variables
X_raw = df_clean[continuous_features + categorical_features]
X_encoded = pd.get_dummies(X_raw, columns=categorical_features, drop_first=True).astype(float)

# Split into Train and Test sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Initialize standard scaler (Z-score normalization: Mean=0, Std=1)
std_scaler = StandardScaler()

# Scaled Dataframes (Only scaling the continuous columns to preserve dummy structure)
X_train_scaled = X_train_raw.copy()
X_test_scaled = X_test_raw.copy()

X_train_scaled[continuous_features] = std_scaler.fit_transform(X_train_raw[continuous_features])
X_test_scaled[continuous_features] = std_scaler.transform(X_test_raw[continuous_features])

print("✅ Data prepared. Features scaled using Z-score Standardization.")

✅ Data prepared. Features scaled using Z-score Standardization.


2. Performing Logistic Regression
We will train a Logistic Regression model on the scaled feature data and evaluate its diagnostic classification breakdown.

In [2]:
# Initialize and fit the Logistic Regression model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Generate Class Predictions and Probability Scores
y_preds = log_reg.predict(X_test_scaled)
y_probs = log_reg.predict_proba(X_test_scaled)[:, 1]

# Display Metrics
print("\n=== LOGISTIC REGRESSION PERFORMANCE METRICS ===")
print(f"Test Accuracy: {accuracy_score(y_test, y_preds):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_preds))

# Feature Importance Tracking (Odds Ratios)
importance_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient (Beta)": log_reg.coef_[0],
    "Odds Ratio": np.exp(log_reg.coef_[0]) # e^beta represents change in odds per unit change
}).sort_values(by="Odds Ratio", ascending=False)

print("\n--- Top Features Influencing High Medical Costs (Odds Ratios) ---")
print(importance_df.head(5).to_string(index=False))


=== LOGISTIC REGRESSION PERFORMANCE METRICS ===
Test Accuracy: 0.7553
ROC-AUC Score: 0.6951

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.96      0.85     14965
           1       0.55      0.16      0.24      5035

    accuracy                           0.76     20000
   macro avg       0.66      0.56      0.55     20000
weighted avg       0.72      0.76      0.70     20000


--- Top Features Influencing High Medical Costs (Odds Ratios) ---
         Feature  Coefficient (Beta)  Odds Ratio
      risk_score            0.820490    2.271613
visits_last_year            0.161529    1.175306
   smoker_Former            0.146491    1.157765
    region_South            0.057352    1.059028
   plan_type_PPO            0.037591    1.038306


3. Automated Strategy Selection DesignAn optimal engineering workflow selects scaling techniques or determines if scaling is fundamentally required based on structural properties (like the presence of extreme outliers or distribution bounds).

- StandardScaler (Z-Score) is best for unbounded variables with outliers because it preserves distribution shapes without compressing variance.
- MinMaxScaler (Normalization) limits values explicitly between $0$ and $1$, making it ideal for algorithms requiring strict boundary distributions.

Below is an automated evaluation module that tests both techniques alongside an unscaled baseline to choose the most robust modeling strategy for the dataset.

In [3]:
def design_optimal_scaling_strategy(X_tr_raw, X_te_raw, y_tr, y_te, cont_cols):
    """
    Evaluates Unscaled vs Standardized vs Normalized data pipelines 
    to choose the optimal deployment strategy based on Test accuracy and ROC-AUC.
    """
    strategies = {}
    
    # 1. Pipeline A: Unscaled Data
    model_raw = LogisticRegression(max_iter=5000, random_state=42).fit(X_tr_raw, y_tr)
    strategies["Unscaled Strategy"] = roc_auc_score(y_te, model_raw.predict_proba(X_te_raw)[:, 1])
    
    # 2. Pipeline B: Standardized Data (Z-Score)
    scaler_s = StandardScaler()
    X_tr_s, X_te_s = X_tr_raw.copy(), X_te_raw.copy()
    X_tr_s[cont_cols] = scaler_s.fit_transform(X_tr_raw[cont_cols])
    X_te_s[cont_cols] = scaler_s.transform(X_te_raw[cont_cols])
    
    model_s = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_s, y_tr)
    strategies["Standardized (Z-score) Strategy"] = roc_auc_score(y_te, model_s.predict_proba(X_te_s)[:, 1])
    
    # 3. Pipeline C: Normalized Data (MinMax)
    scaler_m = MinMaxScaler()
    X_tr_m, X_te_m = X_tr_raw.copy(), X_te_raw.copy()
    X_tr_m[cont_cols] = scaler_m.fit_transform(X_tr_raw[cont_cols])
    X_te_m[cont_cols] = scaler_m.transform(X_te_raw[cont_cols])
    
    model_m = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_m, y_tr)
    strategies["Normalized (MinMax) Strategy"] = roc_auc_score(y_te, model_m.predict_proba(X_te_m)[:, 1])
    
    # Evaluate optimal strategy
    best_strategy = max(strategies, key=strategies.get)
    
    print("\n================ STRATEGIC DESIGN SELECTION ENGINE ================")
    for strategy_name, score in strategies.items():
        print(f"• {strategy_name} -> Test ROC-AUC: {score:.4f}")
        
    print(f"\n🚀 STRATEGIC RECOMMENDATION: Implement the **{best_strategy}** pipeline.")
    
    # Check for outlier vulnerabilities in continuous data to provide engineering context
    skewness = X_tr_raw[cont_cols].skew().abs().max()
    if skewness > 1.5:
        print("CONTEXT NOTE: High skewness/outliers detected. Z-score standardization is mathematically insulated against outlier magnification compared to MinMax.")
    else:
        print("CONTEXT NOTE: Uniform feature distributions allow both bounded normalization and standardization to converge safely.")

# Execute Strategy Brain
design_optimal_scaling_strategy(X_train_raw, X_test_raw, y_train, y_test, continuous_features)

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



================ STRATEGIC DESIGN SELECTION ENGINE ================
• Unscaled Strategy -> Test ROC-AUC: 0.6950
• Standardized (Z-score) Strategy -> Test ROC-AUC: 0.6951
• Normalized (MinMax) Strategy -> Test ROC-AUC: 0.6951

🚀 STRATEGIC RECOMMENDATION: Implement the **Standardized (Z-score) Strategy** pipeline.
CONTEXT NOTE: High skewness/outliers detected. Z-score standardization is mathematically insulated against outlier magnification compared to MinMax.
